# Mutation ↔ Gene Relation-Wise Merge

Merges Mutation–Gene triples from EvoAGE;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [16]:
import os
import pandas as pd

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/MUTATION_GENE/ALL_MUTATION_GENE.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

In [27]:
ENS_2NCBI = pd.read_csv(DB_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv') # 86402 ENS ID's
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()] # 42530 ENS ID's without nan
ENS_2NCBI.head(3)
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
# NCBI_2ENS__dict

del ENS_2NCBI
NCBI_ALL_GENE = pd.read_csv(DB_DIR + 'ncbi/Homo_sapiens.gene_info',sep = '\t') # NCBI_2ENS__dict
# NCBI_ALL_GENEname_dict =  dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict =  dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict =  dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))
NCBI_ALL_GENEIDname_dict
NCBI_ALL_GENE.head(3)

NCBI_ALL_GENE_Syn_Dict =  dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
NCBI_ALL_GENE_Syn_Dict
# Explode into a new dict
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    keys = k.split('|')  # split by '|'
    for single_key in keys:
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[single_key.strip()] = v  # strip just in case
NCBI_ALL_Symb_Desc_dict =  dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))        

In [33]:
NCBI_ALL_GENE

,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type,ENSEMBLE_ID
0,9606,1,A1BG,-,A1B|ABG|GAB|HYST2477,MIM:138670|HGNC:HGNC:5|Ensembl:ENSG00000121410...,19,19q13.43,alpha-1-B glycoprotein,protein-coding,A1BG,alpha-1-B glycoprotein,O,alpha-1B-glycoprotein|HEL-S-163pA|epididymis s...,20250208,-,ENSG00000121410
1,9606,2,A2M,-,A2MD|CPAMD5|FWP007|S863-7,MIM:103950|HGNC:HGNC:7|Ensembl:ENSG00000175899...,12,12p13.31,alpha-2-macroglobulin,protein-coding,A2M,alpha-2-macroglobulin,O,alpha-2-macroglobulin|C3 and PZP-like alpha-2-...,20250208,-,ENSG00000175899
2,9606,3,A2MP1,-,A2MP,HGNC:HGNC:8|Ensembl:ENSG00000291190|AllianceGe...,12,12p13.31,alpha-2-macroglobulin pseudogene 1,pseudo,A2MP1,alpha-2-macroglobulin pseudogene 1,O,pregnancy-zone protein pseudogene,20241210,-,ENSG00000291190
3,9606,9,NAT1,-,AAC1|MNAT|NAT-1|NATI,MIM:108345|HGNC:HGNC:7645|Ensembl:ENSG00000171...,8,8p22,N-acetyltransferase 1,protein-coding,NAT1,N-acetyltransferase 1,O,arylamine N-acetyltransferase 1|N-acetyltransf...,20250208,-,ENSG00000171428
4,9606,10,NAT2,-,AAC2|NAT-2|PNAT,MIM:612182|HGNC:HGNC:7646|Ensembl:ENSG00000156...,8,8p22,N-acetyltransferase 2,protein-coding,NAT2,N-acetyltransferase 2,O,arylamine N-acetyltransferase 2|N-acetyltransf...,20250208,-,ENSG00000156006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193500,741158,8923215,trnD,-,-,-,MT,-,tRNA-Asp,tRNA,-,-,-,-,20200909,-,NaN
193501,741158,8923216,trnP,-,-,-,MT,-,tRNA-Pro,tRNA,-,-,-,-,20200909,-,NaN
193502,741158,8923217,trnA,-,-,-,MT,-,tRNA-Ala,tRNA,-,-,-,-,20200909,-,NaN
193503,741158,8923218,COX1,-,-,-,MT,-,cytochrome c oxidase subunit I,protein-coding,-,-,-,cytochrome c oxidase subunit I,20230818,-,NaN


## 1. Load KG Sources

### 1a. hald

In [7]:
hald = pd.read_csv(PROC_DIR + 'hald/Mutation_Gene_HALD.csv')
hald.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
hald.columns = hald.columns.str.lower()
print(f"hald: {len(hald):,} rows | columns: {list(hald.columns)}")

hald['head_detail_name'] = hald['head']
hald['head_id_is'] = ''
hald['kg_type'] = 'Aging'
hald['species'] = 'Homo species'
hald.head(2)

hald: 313 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'tail_detail_name', 'tail_id_is']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,tail_detail_name,tail_id_is,head_detail_name,head_id_is,kg_type,species
0,rs2805533,Mutation_Gene,ADIPOQ,Mutation,associated,Gene,HALD_KG,"adiponectin, C1Q and collagen domain containing",NCBI_ID,rs2805533,,Aging,Homo species
1,rs2619112,Mutation_Gene,CRP,Mutation,associated,Gene,HALD_KG,C-reactive protein,NCBI_ID,rs2619112,,Aging,Homo species


### 1b. CKG

In [8]:
ckg = pd.read_csv(PROC_DIR + 'CKG/File_11_Mutation_Gene.csv')
ckg.columns = ckg.columns.str.lower()
ckg['head_detail_name'] = ckg['head']
ckg['head_id_is'] = ''
ckg['kg_type'] = 'Generalised'
ckg['species'] = 'Homo species'
ckg 

,head,relation,tail,head_type,tail_type,source,kg_source,tail_detail_name,tail_ensemble_id's,head_id_is,tail_id_is,head_detail_name,kg_type,species
0,chr17:g.30185627A>G,VARIANT_FOUND_IN_GENE,NSRP1,Mutation_Variant,Gene,UniProt,CKG,nuclear speckle splicing regulatory protein 1,ENSG00000126653,,NCBI_Symbol,chr17:g.30185627A>G,Generalised,Homo species
1,chr17:g.30185534C>A,VARIANT_FOUND_IN_GENE,NSRP1,Mutation_Variant,Gene,UniProt,CKG,nuclear speckle splicing regulatory protein 1,ENSG00000126653,,NCBI_Symbol,chr17:g.30185534C>A,Generalised,Homo species
2,chr10:g.60784806A>T,VARIANT_FOUND_IN_GENE,CDK1,Mutation_Variant,Gene,UniProt,CKG,cyclin dependent kinase 1,ENSG00000170312,,NCBI_Symbol,chr10:g.60784806A>T,Generalised,Homo species
3,chr6:g.8429967A>G,VARIANT_FOUND_IN_GENE,SLC35B3,Mutation_Variant,Gene,UniProt,CKG,solute carrier family 35 member B3,ENSG00000124786,,NCBI_Symbol,chr6:g.8429967A>G,Generalised,Homo species
4,chr6:g.8429914T>C,VARIANT_FOUND_IN_GENE,SLC35B3,Mutation_Variant,Gene,UniProt,CKG,solute carrier family 35 member B3,ENSG00000124786,,NCBI_Symbol,chr6:g.8429914T>C,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47154808,chr7:g.33062623T>G,VARIANT_FOUND_IN_GENE,NT5C3A,Mutation_Variant,Gene,UniProt,CKG,"5'-nucleotidase, cytosolic IIIA",ENSG00000122643,,NCBI_Symbol,chr7:g.33062623T>G,Generalised,Homo species
47154809,chr1:g.204987426G>A,VARIANT_FOUND_IN_GENE,NFASC,Mutation_Variant,Gene,UniProt,CKG,neurofascin,ENSG00000163531,,NCBI_Symbol,chr1:g.204987426G>A,Generalised,Homo species
47154810,chr7:g.33021332T>C,VARIANT_FOUND_IN_GENE,NT5C3A,Mutation_Variant,Gene,UniProt,CKG,"5'-nucleotidase, cytosolic IIIA",ENSG00000122643,,NCBI_Symbol,chr7:g.33021332T>C,Generalised,Homo species
47154811,chr7:g.33024094A>C,VARIANT_FOUND_IN_GENE,NT5C3A,Mutation_Variant,Gene,UniProt,CKG,"5'-nucleotidase, cytosolic IIIA",ENSG00000122643,,NCBI_Symbol,chr7:g.33024094A>C,Generalised,Homo species


## 2. Check and Fix Duplicate Columns

In [10]:
SOURCE_DFS = [('ckg', ckg),
             ('hald', hald)]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[ckg] ✓ no duplicates
[hald] ✓ no duplicates


## 3. Align Schemas and Concatenate

In [11]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 47,155,126 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,chr17:g.30185627A>G,VARIANT_FOUND_IN_GENE,NSRP1,Mutation_Variant,NaN,Gene,CKG,,NCBI_Symbol,chr17:g.30185627A>G,nuclear speckle splicing regulatory protein 1,Generalised,Homo species
1,chr17:g.30185534C>A,VARIANT_FOUND_IN_GENE,NSRP1,Mutation_Variant,NaN,Gene,CKG,,NCBI_Symbol,chr17:g.30185534C>A,nuclear speckle splicing regulatory protein 1,Generalised,Homo species


## 4. Standardise Fixed-Value Columns

In [12]:
final_df['head_type'] = 'Mutation'
final_df['tail_type'] = 'Gene'
final_df['relation']  = 'Mutation_Gene'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Mutation_Gene'}
Unique head_id_is: {''}
Unique tail_id_is: {'NCBI_Symbol', 'NCBI_ID'}
Unique kg_source: {'HALD_KG', 'CKG'}


## 5. Deduplicate and Add Schema Columns

In [18]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
final_df_dedup.head()

After deduplication: 13,638,792 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,START_ID,Mutation_Gene,END_ID,Mutation,None,Gene,CKG,,NCBI_Symbol,START_ID,None,Generalised,Homo species
1,chr10:g.100042434G>A,Mutation_Gene,CPNE1,Mutation,None,Gene,CKG,,NCBI_Symbol,chr10:g.100042434G>A,copine 1,Generalised,Homo species
2,chr10:g.100042434G>C,Mutation_Gene,CPNE1,Mutation,None,Gene,CKG,,NCBI_Symbol,chr10:g.100042434G>C,copine 1,Generalised,Homo species
3,chr10:g.100042434G>T,Mutation_Gene,CPNE1,Mutation,None,Gene,CKG,,NCBI_Symbol,chr10:g.100042434G>T,copine 1,Generalised,Homo species
4,chr10:g.100042435G>C,Mutation_Gene,CPNE1,Mutation,None,Gene,CKG,,NCBI_Symbol,chr10:g.100042435G>C,copine 1,Generalised,Homo species


## 6. QC — NaN Check

In [14]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,13638515,0,13638515
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [40]:
len(set(final_df_dedup[final_df_dedup['tail_detail_name'].isna()])) # These 13 Gene names are not findable in ncbi 

13

In [41]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is', 'kg_type','species']:
    print(f"{col:20s}: {set(final_df_dedup[col])}")

relation            : {'Mutation_Gene'}
head_type           : {'Mutation'}
tail_type           : {'Gene'}
relation_type       : {'lead', 'exhibit', 'recruit', 'disrupt', 'affect', 'investigate', 'implicated', 'correspond', 'considered', 'tend', 'modify', 'compared', 'rescue', 'demonstrate', 'induce', 'link', 'identified', 'create', 'related', 'significant effect on', 'increase', 'enhance', 'exceed', 'result', 'regulate', 'predict', 'situated', 'identify', 'compare', 'take', 'act', 'selected', 'colocalize', 'alter', 'determined', 'influence', 'be with', 'created', 'emerge', 'of allele be', 'produce', 'involve', 'potentiate', 'display', 'reach', 'expression of', 'vary', 'tag', 'found', 'encode', 'include', 'draw', 'pseudophosphorylate', 'confer', 'show', 'characterized', 'measure', 'included', None, 'cover', 'correlated', 'examine', 'known', 'associated', 'lie', 'change', 'treated', 'transactivate', 'belong', 'detected', 'suggest', 'cause', 'abilities of', 'be in', 'bear', 'be early pred

## 7. Save Output

In [42]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 13,638,792 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/MUTATION_GENE/ALL_MUTATION_GENE.csv


In [ ]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

In [44]:
final_df_dedup.to_parquet('/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/MUTATION_GENE/ALL_MUTATION_GENE.parquet')